# Train GAM đối chiếu — 3 nhánh, có chọn λ, có lưu artifact

> **Thay thế bản cũ.** Bản trước chỉ train nhánh *giá cơ bản* và **không lưu file nào** — nên kết
> quả GAM trong báo cáo không tái tạo được từ repo. Bản này sửa cả hai.

## Bản này khác gì bản cũ

| | Bản cũ | Bản này |
|---|---|---|
| Số nhánh | 1 (giá cơ bản) | **3** (giá cơ bản · hệ số nhân · giá trực tiếp) |
| `λ` (độ phạt làm mượt) | mặc định 0,6 | **chọn bằng gridsearch** |
| Lưu model | ❌ không | ✅ `GAM/*.joblib` |
| Lưu dự đoán | ❌ không | ✅ `evaluation/pred_gam.parquet` |

## GAM là gì ở đây

```
Hồi quy tuyến tính:  y = b₀ + b₁·x₁ + b₂·x₂ + ...      ← mỗi biến 1 đường THẲNG
GAM:                 y = b₀ + f₁(x₁) + f₂(x₂) + ...    ← mỗi biến 1 đường CONG riêng
Boosting tree:       y = f(x₁, x₂, ...)                 ← học cả TƯƠNG TÁC, hộp đen
```

Mỗi `fᵢ` là spline ghép từ `n_splines` đoạn trơn, phạt độ gấp khúc bằng `λ`. Các hàm **cộng dồn**,
không nhân ⇒ GAM **không bắt được tương tác** giữa các feature.

## Kiến trúc 3 nhánh

| Nhánh | Target | Feature | Term |
|---|---|---|---|
| A — giá cơ bản | `log(base_price)` | `B_NUM` (10) + `CAT` (4) | 14 |
| B — hệ số nhân | `log(multiplier)` | `M_NUM` (7) + `CAT` (4) | 11 |
| C — giá trực tiếp | `log(shown_price)` | `D_NUM` (10) + `CAT` (4) | 14 |

Train **riêng từng tháng** (01/02/03), cùng feature contract và giao thức chia dữ liệu với nhóm
boosting tree ⇒ so sánh công bằng.

⏱️ **Thời gian chạy ~20–25 phút** (gridsearch + 9 lần fit trên ~1,55 triệu dòng mỗi lần).

In [1]:
import warnings, time, joblib
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from pathlib import Path
from pygam import LinearGAM, s, f
from sklearn.preprocessing import OrdinalEncoder

CAT = ["service_name", "pickup_location_name", "dropoff_location_name", "weather_main"]
B_NUM = ["quote_distance", "quote_duration", "gio_vn", "latest_observed_base",
         "history_60m_price_mean", "history_60m_price_std",
         "history_60m_price_slope_per_minute", "latest_observed_quote_distance",
         "latest_observed_quote_duration", "actual_observation_age_minutes"]
D_NUM = ["quote_distance", "quote_duration", "gio_vn", "latest_observed_price",
         "history_60m_price_mean", "history_60m_price_std",
         "history_60m_price_slope_per_minute", "latest_observed_quote_distance",
         "latest_observed_quote_duration", "actual_observation_age_minutes"]
M_NUM = ["pricing_market_imbalance_5m_lag", "pricing_demand_index_5m_lag",
         "pricing_supply_index_5m_lag", "pricing_quote_count_5m_lag",
         "latest_observed_multiplier", "gio_vn", "actual_observation_age_minutes"]

N_SPLINES = 12
LAM_GRID = np.logspace(-2, 2, 5)      # 0,01 · 0,1 · 1 · 10 · 100
N_MAU_TIM_LAM = 120_000               # co mau con de gridsearch cho nhanh

GAMDIR = Path("../GAM"); GAMDIR.mkdir(exist_ok=True)
EVALDIR = Path("../evaluation"); EVALDIR.mkdir(exist_ok=True)
PREP = Path("../../data/hcm_train_ready.parquet")
assert PREP.exists(), "Chua co hcm_train_ready.parquet -> chay 00_chuan_bi_du_lieu.ipynb truoc!"

COLS = list(dict.fromkeys(CAT + B_NUM + D_NUM + M_NUM +
                          ["target_shown_price", "target_shown_multiplier",
                           "latest_observed_price", "latest_observed_multiplier",
                           "evaluation_month", "split"]))
COLS = [c for c in COLS if c != "latest_observed_base"]      # cot dan xuat
df = pd.read_parquet(PREP, columns=COLS)

df["base_price"] = df.target_shown_price / df.target_shown_multiplier.clip(lower=0.1)
df["latest_observed_base"] = df.latest_observed_price / df.latest_observed_multiplier.clip(lower=0.1)

enc = OrdinalEncoder(dtype=np.int64, handle_unknown="use_encoded_value", unknown_value=-1)
df[CAT] = enc.fit_transform(df[CAT].astype(str))
joblib.dump(enc, GAMDIR / "encoder.joblib")

print(f"Nap {len(df):,} dong | {len(CAT)} categorical + {len(set(B_NUM+D_NUM+M_NUM))} numeric")
for i, c in enumerate(CAT):
    print(f"  {c:24s} {len(enc.categories_[i])} nhom")
print(f"\nDa luu encoder -> {GAMDIR / 'encoder.joblib'}")

Nap 6,897,051 dong | 4 categorical + 16 numeric
  service_name             2 nhom
  pickup_location_name     3 nhom
  dropoff_location_name    3 nhom
  weather_main             4 nhom

Da luu encoder -> ..\GAM\encoder.joblib


## 1. Chọn `λ` bằng gridsearch

`λ` là **độ phạt gấp khúc**: nhỏ thì đường cong tự do (dễ overfit), lớn thì bị ép về gần đường
thẳng (underfit). Bản cũ dùng mặc định 0,6 mà không kiểm chứng.

Gridsearch chạy trên **mẫu con 120.000 dòng** cho nhanh, chọn theo **GCV** (Generalized Cross
Validation — xấp xỉ leave-one-out mà không phải chạy lại nhiều lần). `λ` chọn xong dùng chung cho
cả 3 tháng.

In [2]:
def tao_terms(n_num, n_cat):
    """n_num smooth term dau + n_cat factor term sau."""
    t = s(0, n_splines=N_SPLINES)
    for i in range(1, n_num):
        t += s(i, n_splines=N_SPLINES)
    for i in range(n_num, n_num + n_cat):
        t += f(i)
    return t


NHANH = {
    "gia_co_ban":  dict(num=B_NUM, target=lambda d: d.base_price),
    "heso":        dict(num=M_NUM, target=lambda d: d.target_shown_multiplier),
    "gia":         dict(num=D_NUM, target=lambda d: d.target_shown_price),
}

rng = np.random.default_rng(42)
tr_all = df[df.split == "train"]
mau = tr_all.iloc[rng.choice(len(tr_all), min(N_MAU_TIM_LAM, len(tr_all)), replace=False)]

LAM_CHON = {}
for ten, cfg in NHANH.items():
    feats = cfg["num"] + CAT
    X = mau[feats].values.astype(float)
    y = np.log(np.clip(cfg["target"](mau).values, 1e-6, None))
    t0 = time.time()
    g = LinearGAM(tao_terms(len(cfg["num"]), len(CAT)), max_iter=25)
    try:
        g.gridsearch(X, y, lam=LAM_GRID, progress=False)
        lam = float(np.ravel(g.lam)[0])
    except Exception as e:
        lam = 0.6
        print(f"  [{ten}] gridsearch loi ({type(e).__name__}) -> dung mac dinh 0.6")
    LAM_CHON[ten] = lam
    print(f"  [{ten:11s}] lam = {lam:>8.3f} | GCV {g.statistics_.get('GCV', float('nan')):.5f} "
          f"| {time.time()-t0:.0f}s")

print()
print("Lam da chon:", {k: round(v, 3) for k, v in LAM_CHON.items()})
print("(ban cu dung mac dinh 0.6 cho tat ca)")

  [gia_co_ban ] lam =    1.000 | GCV 0.03349 | 25s


  [heso       ] lam =    0.010 | GCV 0.00106 | 16s


  [gia        ] lam =    0.100 | GCV 0.03804 | 24s

Lam da chon: {'gia_co_ban': 1.0, 'heso': 0.01, 'gia': 0.1}
(ban cu dung mac dinh 0.6 cho tat ca)


## 2. Train 3 nhánh × 3 tháng và lưu model

`f()` của pygam chỉ nhận giá trị categorical **đã xuất hiện trong tập train**. Với dữ liệu đầy đủ
(~1,55 triệu dòng/tháng) mọi nhóm đều có mặt, nhưng vẫn lọc và báo rõ để an toàn.

In [3]:
thangs = sorted(df.evaluation_month.unique())
models = {ten: {} for ten in NHANH}
print("Train theo thang:", thangs)
tong_t0 = time.time()

for ten, cfg in NHANH.items():
    feats = cfg["num"] + CAT
    print(f"\n[{ten}]  {len(cfg['num'])} smooth + {len(CAT)} factor = {len(feats)} term "
          f"| lam = {LAM_CHON[ten]:.3f}")
    for th in thangs:
        sub = df[df.evaluation_month == th]
        tr = sub[sub.split == "train"]
        t0 = time.time()
        X = tr[feats].values.astype(float)
        y = np.log(np.clip(cfg["target"](tr).values, 1e-6, None))
        g = LinearGAM(tao_terms(len(cfg["num"]), len(CAT)),
                      lam=LAM_CHON[ten], max_iter=25).fit(X, y)
        models[ten][th] = g
        print(f"   [{th}] n_train={len(tr):,} | "
              f"pseudo-R2 {g.statistics_['pseudo_r2']['explained_deviance']:.4f} | "
              f"{time.time()-t0:.0f}s")
    joblib.dump(models[ten], GAMDIR / f"{ten}.joblib")
    print(f"   -> da luu {GAMDIR / (ten + '.joblib')}")

print(f"\nTONG THOI GIAN TRAIN: {(time.time()-tong_t0)/60:.1f} phut")

Train theo thang: ['2026-01', '2026-02', '2026-03']

[gia_co_ban]  10 smooth + 4 factor = 14 term | lam = 1.000


   [2026-01] n_train=1,544,286 | pseudo-R2 0.6908 | 94s


   [2026-02] n_train=1,547,985 | pseudo-R2 0.6919 | 86s


   [2026-03] n_train=1,549,528 | pseudo-R2 0.6910 | 95s
   -> da luu ..\GAM\gia_co_ban.joblib

[heso]  7 smooth + 4 factor = 11 term | lam = 0.010


   [2026-01] n_train=1,544,286 | pseudo-R2 0.9531 | 50s


   [2026-02] n_train=1,547,985 | pseudo-R2 0.9548 | 52s


   [2026-03] n_train=1,549,528 | pseudo-R2 0.9526 | 53s
   -> da luu ..\GAM\heso.joblib

[gia]  10 smooth + 4 factor = 14 term | lam = 0.100


   [2026-01] n_train=1,544,286 | pseudo-R2 0.7291 | 83s


   [2026-02] n_train=1,547,985 | pseudo-R2 0.7306 | 96s


   [2026-03] n_train=1,549,528 | pseudo-R2 0.7274 | 84s
   -> da luu ..\GAM\gia.joblib

TONG THOI GIAN TRAIN: 11.7 phut


## 3. Sinh dự đoán trên tập test và lưu parquet

⚠️ **Thứ tự hàng:** gộp theo **tháng** (`concat` từng tháng theo thứ tự tăng dần). Các notebook
đánh giá dựa vào thứ tự này để khớp với `uq_pred_test.parquet` — không được đổi.

In [4]:
phan = []
for th in thangs:
    sub = df[df.evaluation_month == th]
    tr = sub[sub.split == "train"]
    te = sub[sub.split == "test"].copy()

    hop_le = np.ones(len(te), dtype=bool)
    for c in CAT:
        hop_le &= te[c].isin(set(tr[c].unique())).values
    if not hop_le.all():
        print(f"  [{th}] CANH BAO: bo {(~hop_le).sum():,}/{len(te):,} dong test "
              f"co nhom categorical khong xuat hien trong train")
        te = te[hop_le].copy()

    out = pd.DataFrame({
        "evaluation_month": te.evaluation_month.values,
        "split": "test",
        "algo": "GAM",
        "gia_that": te.target_shown_price.values.astype("float64"),
    })
    for ten, cot in [("gia_co_ban", "base_pred"), ("heso", "heso_pred"),
                     ("gia", "truc_tiep_pred")]:
        feats = NHANH[ten]["num"] + CAT
        out[cot] = np.exp(models[ten][th].predict(te[feats].values.astype(float)))
    out["hybrid_pred"] = out.base_pred * out.heso_pred
    phan.append(out)
    print(f"  [{th}] du doan xong {len(out):,} dong")

pred = pd.concat(phan, ignore_index=True)
pred = pred[["evaluation_month", "split", "algo", "gia_that",
             "base_pred", "heso_pred", "hybrid_pred", "truc_tiep_pred"]]
pred.to_parquet(EVALDIR / "pred_gam.parquet", index=False)
print(f"\nDa luu {len(pred):,} dong -> {EVALDIR / 'pred_gam.parquet'}")
print(pred.head(3).to_string())

  [2026-01] du doan xong 315,360 dong


  [2026-02] du doan xong 234,632 dong


  [2026-03] du doan xong 314,368 dong



Da luu 864,360 dong -> ..\evaluation\pred_gam.parquet
  evaluation_month split algo  gia_that     base_pred  heso_pred    hybrid_pred  truc_tiep_pred
0          2026-01  test  GAM  106000.0  87239.170812   1.169984  102068.406814   101292.675070
1          2026-01  test  GAM  106000.0  87220.488724   1.184144  103281.639733   100443.268500
2          2026-01  test  GAM  106000.0  87358.033685   1.176994  102819.853345   103962.239649


## 4. Đánh giá — GAM vs boosting tree

In [5]:
uq = pd.read_parquet(EVALDIR / "uq_pred_test.parquet")
assert len(uq) == len(pred), f"So dong lech: uq {len(uq):,} vs pred {len(pred):,}"
assert np.allclose(uq.gia_that.values, pred.gia_that.values), \
    "THU TU HANG KHONG KHOP voi uq_pred_test.parquet!"
print("Khop hang voi uq_pred_test.parquet — cac notebook danh gia dung duoc ngay.")


def do(p, t):
    p, t = np.asarray(p, float), np.asarray(t, float)
    return dict(MAE=np.abs(p-t).mean(), MAPE=(np.abs(p-t)/t).mean(),
                R2=1 - ((p-t)**2).sum()/((t-t.mean())**2).sum())


base_that = uq.base_that.values
heso_that = uq.heso_that.values
rows = []
for nhanh, pg, pc, tt in [
        ("Giá cơ bản", pred.base_pred, uq.base_pred, base_that),
        ("Hệ số nhân", pred.heso_pred, uq.heso_pred, heso_that),
        ("Hybrid (A×B)", pred.hybrid_pred, uq.hybrid_pred, uq.gia_that.values)]:
    a, b = do(pg, tt), do(pc, tt)
    rows.append({"Nhánh": nhanh, "GAM MAE": round(a["MAE"], 4),
                 "HistGB MAE": round(b["MAE"], 4),
                 "Chênh": f"{(a['MAE']/b['MAE']-1)*100:+.1f}%",
                 "GAM R²": round(a["R2"], 4), "HistGB R²": round(b["R2"], 4)})

pg_ = pd.read_parquet(EVALDIR / "pred_gia.parquet",
                      columns=["split", "algo", "pred", "target_shown_price"])
pg_ = pg_[(pg_.split == "test") & (pg_.algo == "XGBoost")]
a = do(pred.truc_tiep_pred, pred.gia_that); b = do(pg_.pred, pg_.target_shown_price)
rows.append({"Nhánh": "Giá trực tiếp", "GAM MAE": round(a["MAE"], 4),
             "HistGB MAE": round(b["MAE"], 4),
             "Chênh": f"{(a['MAE']/b['MAE']-1)*100:+.1f}%",
             "GAM R²": round(a["R2"], 4), "HistGB R²": round(b["R2"], 4)})

KQ = pd.DataFrame(rows)
print("\nKET QUA (cot HistGB MAE o dong cuoi la XGBoost — model truc tiep tot nhat):")
display(KQ)

# So voi ban cu (lam mac dinh 0.6) de biet gridsearch co giup khong
CU = {"Giá cơ bản": 15098.68, "Hệ số nhân": 0.0302,
      "Hybrid (A×B)": 18237.58, "Giá trực tiếp": 19170.15}
print("\nSO VOI BAN CU (lam mac dinh 0.6):")
for _, r in KQ.iterrows():
    cu = CU[r["Nhánh"]]
    d = (r["GAM MAE"]/cu - 1)*100
    dau = "TOT HON" if d < -0.05 else ("te hon" if d > 0.05 else "nhu cu")
    print(f"  {r['Nhánh']:16s} {cu:>10.4f} -> {r['GAM MAE']:>10.4f}  ({d:+.2f}%)  {dau}")

Khop hang voi uq_pred_test.parquet — cac notebook danh gia dung duoc ngay.



KET QUA (cot HistGB MAE o dong cuoi la XGBoost — model truc tiep tot nhat):


,Nhánh,GAM MAE,HistGB MAE,Chênh,GAM R²,HistGB R²
0,Giá cơ bản,15098.7308,15030.2181,+0.5%,0.6599,0.6564
1,Hệ số nhân,0.0303,0.0232,+30.5%,0.9400,0.9609
2,Hybrid (A×B),18241.7162,18044.6671,+1.1%,0.7286,0.7299
3,Giá trực tiếp,19172.5255,18806.7164,+1.9%,0.6977,0.7056



SO VOI BAN CU (lam mac dinh 0.6):
  Giá cơ bản       15098.6800 -> 15098.7308  (+0.00%)  nhu cu
  Hệ số nhân           0.0302 ->     0.0303  (+0.33%)  te hon
  Hybrid (A×B)     18237.5800 -> 18241.7162  (+0.02%)  nhu cu
  Giá trực tiếp    19170.1500 -> 19172.5255  (+0.01%)  nhu cu


## 5. Kết luận

### Đọc kết quả thế nào

| Nếu | Nghĩa |
|---|---|
| GAM **gần bằng** boosting tree | Quan hệ feature ↔ target là **cộng dồn thuần** — GAM đáng dùng vì dễ giải thích hơn nhiều |
| GAM **kém rõ rệt** | Có **tương tác** giữa các feature mà mô hình cộng dồn không bắt được ⇒ phải giữ boosting tree |

Kết quả tuần 3 (với `λ` mặc định): giá cơ bản chênh **+0,5%**, hệ số nhân chênh **+30%** — và đã
kiểm chứng trực tiếp bằng cách cho cây học lại phần dư của GAM (R² = 0,554 ở hệ số nhân, 0,018 ở
giá cơ bản). Xem `evaluation/08_truc_quan_GAM.ipynb`.

### Artifact sinh ra

| File | Nội dung |
|---|---|
| `GAM/encoder.joblib` | `OrdinalEncoder` cho 4 cột categorical |
| `GAM/gia_co_ban.joblib` | dict `{tháng: LinearGAM}` — nhánh A |
| `GAM/heso.joblib` | dict `{tháng: LinearGAM}` — nhánh B |
| `GAM/gia.joblib` | dict `{tháng: LinearGAM}` — nhánh C |
| `evaluation/pred_gam.parquet` | Dự đoán 4 nhánh trên tập test, khớp hàng với `uq_pred_test.parquet` |

### Việc tiếp theo

**GAM trên transformed feature space** — gợi ý của mentor tuần 2, vẫn chưa làm. Ý tưởng: nếu biến
đổi feature khéo (ví dụ tạo sẵn `quãng_đường × cao_điểm`), mô hình cộng dồn có thể bắt được một
phần tương tác mà không mất tính giải thích được.

### Liên quan

| File | Vai trò |
|---|---|
| `../evaluation/08_truc_quan_GAM.ipynb` | Trực quan kết quả: đường cong, p-value, bằng chứng tương tác |
| `../evaluation/04_eval_hybrid.ipynb` | Ghép A × B cho nhóm boosting tree |
| `_common_train.py` | Định nghĩa `CAT`, `B_NUM`, `D_NUM`, `M_NUM` |